# 03 — Telemetry risk model

This notebook uses the weekly telemetry table to predict whether a generator will fail within the next 30 days.

## Features included
- days since last PM
- runtime hours
- load
- oil temperature
- coolant temperature
- battery voltage
- vibration
- fuel rate
- alarm count
- anomaly score

## Label
- `failure_within_30d`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = Path('..')
DATA = BASE / 'data' / 'processed'
OUT = BASE / 'outputs'

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier


In [ ]:
df = pd.read_csv(DATA / 'telemetry_weekly.csv', parse_dates=['timestamp'])
df.head()

In [ ]:
features = [
    'age_years',
    'days_since_last_pm',
    'runtime_hours_week',
    'avg_load_pct',
    'oil_temp_c',
    'coolant_temp_c',
    'battery_voltage',
    'vibration_mm_s',
    'fuel_rate_lph',
    'alarm_count',
    'anomaly_score',
]
target = 'failure_within_30d'

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=160,
    max_depth=10,
    min_samples_leaf=8,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))
print(classification_report(y_test, pred))

## Feature importance

This chart helps you explain the model to a non-technical audience.

In [ ]:
imp = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
plt.barh(imp['feature'], imp['importance'])
plt.gca().invert_yaxis()
plt.title('Random Forest feature importance')
plt.tight_layout()
plt.show()

imp

In [ ]:
sample = df.sample(6000, random_state=42)
plt.figure(figsize=(8, 5))
plt.scatter(
    sample['vibration_mm_s'],
    sample['battery_voltage'],
    s=8,
    alpha=0.4,
    c=sample['failure_within_30d']
)
plt.title('Battery voltage vs vibration')
plt.xlabel('Vibration (mm/s)')
plt.ylabel('Battery voltage')
plt.tight_layout()
plt.show()